# higher_order_functions
**Prerequisites:** function_basics, parameters_and_arguments, return_values, scope, closures, lambda_functions

**Outcome:** After this notebook you will know exactly what makes a function higher-order, how to write functions that accept and return other functions, and how this pattern produces flexible, reusable, composable code without repetition.

## The Problem

In [ ]:
# three pipeline stages — each does logging, timing, and error handling
# the scaffolding is identical, only the core logic differs

import time

def extract(records):
    print("extract: started")
    start = time.perf_counter()
    try:
        result = [{**r, "source": "db"} for r in records]
    except Exception as e:
        print(f"extract: failed — {e}")
        return None
    elapsed = time.perf_counter() - start
    print(f"extract: done in {elapsed:.4f}s")
    return result

def transform(records):
    print("transform: started")
    start = time.perf_counter()
    try:
        result = [{**r, "status": "transformed"} for r in records]
    except Exception as e:
        print(f"transform: failed — {e}")
        return None
    elapsed = time.perf_counter() - start
    print(f"transform: done in {elapsed:.4f}s")
    return result

# every new stage needs the same 8 lines of scaffolding
# if you change how timing works you update it in every function

## The Concept

A higher-order function is a function that either accepts another function as an argument, returns a function, or both. This is possible in Python because functions are first-class objects — they can be stored in variables, passed as arguments, and returned like any other value. Higher-order functions let you separate what to do from how to do it, inject behaviour into existing code without modifying it, and eliminate structural repetition by abstracting the pattern that surrounds the logic that actually changes.

## Minimal Example

In [ ]:
# accepts a function as argument
def apply(func, value):
    return func(value)

print(apply(str.upper, "api"))       # API
print(apply(len, [1, 2, 3, 4, 5]))   # 5
print(apply(lambda x: x * 2, 10))    # 20

## How Python Does It

When you pass a function as an argument Python passes a reference to the function object — no copy, no special wrapping. Inside the receiving function that reference is bound to a parameter name and can be called with `()` exactly like any other callable. When a function returns a function it returns a reference to the inner function object. The caller receives that reference and can store it, call it, or pass it on. There is no mechanism specific to higher-order functions — it is the same object model used for everything else.

In [ ]:
def run(func, data):
    print(f"calling {func.__name__} with {len(data)} items")
    return func(data)

def summarise(records):
    return {"count": len(records), "ids": [r["id"] for r in records]}

records = [{"id": "job_1"}, {"id": "job_2"}, {"id": "job_3"}]
result  = run(summarise, records)
print(result)

## Building Up

In [ ]:
# returning a function — function factory
def make_validator(required_fields):
    def validate(record):
        missing = [f for f in required_fields if not record.get(f)]
        return {"ok": len(missing) == 0, "missing": missing}
    return validate

validate_job     = make_validator(["id", "status", "region"])
validate_minimal = make_validator(["id"])

print(validate_job({"id": "job_1", "status": "pending"}))
print(validate_minimal({"id": "job_2"}))

In [ ]:
# applying a list of functions in sequence
def pipeline(data, *steps):
    for step in steps:
        data = step(data)
    return data

def remove_empty(records):
    return [r for r in records if r.get("id")]

def tag_processed(records):
    return [{**r, "processed": True} for r in records]

def sort_by_id(records):
    return sorted(records, key=lambda r: r["id"])

raw = [
    {"id": "job_3", "status": "done"},
    {"id": "",      "status": "pending"},
    {"id": "job_1", "status": "failed"},
]

result = pipeline(raw, remove_empty, tag_processed, sort_by_id)
for r in result:
    print(r)

In [ ]:
# storing functions in a dict — dispatch table
def handle_done(record):
    return {**record, "archived": True}

def handle_failed(record):
    return {**record, "retry": True}

def handle_pending(record):
    return record

handlers = {
    "done":    handle_done,
    "failed":  handle_failed,
    "pending": handle_pending,
}

records = [
    {"id": "job_1", "status": "done"},
    {"id": "job_2", "status": "failed"},
    {"id": "job_3", "status": "pending"},
]

for record in records:
    handler = handlers.get(record["status"], handle_pending)
    print(handler(record))

In [ ]:
# wrapping a function — add behaviour before and after without touching the original
import time

def timed(func):
    def wrapper(*args, **kwargs):
        start  = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} completed in {elapsed:.4f}s")
        return result
    return wrapper

def extract(records):
    return [{**r, "source": "db"} for r in records]

timed_extract = timed(extract)

records = [{"id": f"job_{i}"} for i in range(1000)]
result  = timed_extract(records)
print(f"got {len(result)} records")

In [ ]:
# function composition — build a new function from two existing ones
def compose(f, g):
    def composed(*args, **kwargs):
        return f(g(*args, **kwargs))
    return composed

def strip(text):
    return text.strip()

def upper(text):
    return text.upper()

clean_upper = compose(upper, strip)

print(clean_upper("  api  "))   # API
print(clean_upper("  worker"))  # WORKER

In [ ]:
# passing a function to control branching behaviour
def process_records(records, on_valid, on_invalid):
    for record in records:
        if record.get("id") and record.get("status"):
            on_valid(record)
        else:
            on_invalid(record)

def archive(record):
    print(f"archiving  {record['id']}")

def quarantine(record):
    print(f"quarantine {record.get('id', 'no-id')}")

records = [
    {"id": "job_1", "status": "done"},
    {"id": "",      "status": "pending"},
    {"id": "job_3", "status": "failed"},
]

process_records(records, on_valid=archive, on_invalid=quarantine)

## Where It Breaks

In [ ]:
# passing the result of calling a function instead of the function itself
def double(x):
    return x * 2

def apply(func, value):
    return func(value)

# wrong — double(5) evaluates to 10, then apply tries to call 10
try:
    result = apply(double(5), 3)
except TypeError as e:
    print(e)

# correct — pass the function object without calling it
result = apply(double, 3)
print(result)   # 6

In [ ]:
# wrapper that swallows the return value
def logged(func):
    def wrapper(*args, **kwargs):
        print(f"calling {func.__name__}")
        func(*args, **kwargs)   # bug: return value discarded
    return wrapper

def get_count(records):
    return len(records)

logged_count = logged(get_count)
result = logged_count([1, 2, 3])
print(result)   # None — return value lost

## The Fix

In [ ]:
# always return the result from the wrapper
def logged(func):
    def wrapper(*args, **kwargs):
        print(f"calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result!r}")
        return result          # always return
    return wrapper

def get_count(records):
    return len(records)

logged_count = logged(get_count)
result = logged_count([1, 2, 3])
print(result)   # 3

## In a Real System

In [ ]:
# solving the original problem — scaffolding extracted into a higher-order function
import time

def with_observability(stage_name, func):
    def wrapper(records):
        print(f"{stage_name}: started with {len(records)} records")
        start = time.perf_counter()
        try:
            result = func(records)
        except Exception as e:
            print(f"{stage_name}: failed — {e}")
            return None
        elapsed = time.perf_counter() - start
        print(f"{stage_name}: done in {elapsed:.4f}s — {len(result)} records out")
        return result
    return wrapper

# core logic — no scaffolding
def _extract(records):
    return [{**r, "source": "db"} for r in records]

def _transform(records):
    return [{**r, "status": "transformed"} for r in records]

def _load(records):
    return [{**r, "persisted": True} for r in records]

# wrap with observability once at setup time
extract   = with_observability("extract",   _extract)
transform = with_observability("transform", _transform)
load      = with_observability("load",      _load)

raw = [{"id": f"job_{i}"} for i in range(5)]
result = load(transform(extract(raw)))
print(f"\nfinal: {len(result)} records")

## Performance

Passing a function as an argument adds one reference operation — effectively zero cost. The wrapper pattern adds one extra function call per invocation — a small fixed overhead. In extremely hot loops called millions of times per second, avoid wrapping and call the inner function directly. For anything else the overhead is irrelevant and the structural clarity is worth far more than the nanoseconds saved.

## Anti-Pattern

In [ ]:
# anti-pattern: using if/elif chains instead of a dispatch table
def handle(record):
    if record["status"] == "done":
        return {**record, "archived": True}
    elif record["status"] == "failed":
        return {**record, "retry": True}
    elif record["status"] == "pending":
        return record
    # adding a new status requires editing this function

# correct: dispatch table — adding a status means adding one dict entry
def handle_done(r):    return {**r, "archived": True}
def handle_failed(r):  return {**r, "retry": True}
def handle_pending(r): return r

HANDLERS = {
    "done":    handle_done,
    "failed":  handle_failed,
    "pending": handle_pending,
}

def handle(record):
    handler = HANDLERS.get(record["status"], handle_pending)
    return handler(record)

print(handle({"id": "job_1", "status": "done"}))
print(handle({"id": "job_2", "status": "failed"}))

## Interview Signals

- What makes a function higher-order?
- What is a dispatch table and why is it preferable to a long if/elif chain?
- What is function composition and how do you implement it in Python?
- Why must a wrapper function always return the result of the wrapped call?

## Exercise

In [ ]:
def make_pipeline(*steps):
    """
    Takes any number of single-argument functions.
    Returns a new function that applies them left to right:
    the output of each step becomes the input of the next.

    If any step returns None the pipeline must stop immediately
    and return None — do not pass None to the next step.

    Bug: the function body is missing — implement it.
    """
    pass


def remove_empty(records):
    return [r for r in records if r.get("id")]

def tag(records):
    return [{**r, "tagged": True} for r in records]

def sort_by_id(records):
    return sorted(records, key=lambda r: r["id"])

def fail_if_empty(records):
    return None if len(records) == 0 else records


run = make_pipeline(remove_empty, tag, sort_by_id)

raw = [
    {"id": "job_3"},
    {"id": ""},
    {"id": "job_1"},
]

result = run(raw)

assert result is not None
assert len(result) == 2
assert result[0]["id"] == "job_1"
assert result[1]["id"] == "job_3"
assert all(r["tagged"] for r in result)

# pipeline stops at None
run_with_fail = make_pipeline(remove_empty, fail_if_empty, tag)
assert run_with_fail([{"id": ""}]) is None   # all empty — fail_if_empty returns None

# empty steps — identity pipeline
identity = make_pipeline()
assert identity([1, 2, 3]) == [1, 2, 3]

print("all assertions passed")